In [3]:
from google.cloud import storage
from google.cloud import bigquery

storage_client = storage.Client(project="urban-intelligence-pipeline-sv")
buckets = list(storage_client.list_buckets())
print("GCS Buckets found:")
for bucket in buckets:
    print(f"  → {bucket.name}")

bq_client = bigquery.Client(project="urban-intelligence-pipeline-sv")
datasets = list(bq_client.list_datasets())
print("\nBigQuery Datasets found:")
for dataset in datasets:
    print(f"  → {dataset.dataset_id}")

print("\n✅ GCP Connection successful!")

GCS Buckets found:
  → urban-intelligence-pipeline-sv-raw
  → urban-intelligence-pipeline-sv-scripts
  → urban-intelligence-pipeline-sv-staging
  → urban-intelligence-tf-state-sv

BigQuery Datasets found:
  → marts
  → raw
  → staging

✅ GCP Connection successful!


In [4]:
# Cell 2 — Imports and Configuration
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from google.cloud import storage, bigquery
from datetime import datetime, date
import os
import io
import warnings
warnings.filterwarnings('ignore')

# Configuration — all in one place
PROJECT_ID = "urban-intelligence-pipeline-sv"
RAW_BUCKET = "urban-intelligence-pipeline-sv-raw"
INGESTION_DATE = datetime.now().strftime("%Y-%m-%d")

# Initialize clients
storage_client = storage.Client(project=PROJECT_ID)
bq_client = bigquery.Client(project=PROJECT_ID)

print(f"✅ Config loaded — Ingestion date: {INGESTION_DATE}")

✅ Config loaded — Ingestion date: 2026-04-28


In [13]:
pip install db-dtypes

Note: you may need to restart the kernel to use updated packages.


In [10]:
# Cell 3 — Extract NYC Taxi Data (correct schema)
print("🚕 Querying NYC Taxi public dataset...")

taxi_query = """
SELECT
    vendor_id,
    pickup_datetime,
    dropoff_datetime,
    passenger_count,
    trip_distance,
    rate_code,
    payment_type,
    fare_amount,
    extra,
    mta_tax,
    tip_amount,
    tolls_amount,
    imp_surcharge,
    airport_fee,
    total_amount,
    pickup_location_id,
    dropoff_location_id,
    data_file_year,
    data_file_month,
    DATE(pickup_datetime)                   AS trip_date,
    EXTRACT(HOUR FROM pickup_datetime)      AS pickup_hour,
    EXTRACT(DAYOFWEEK FROM pickup_datetime) AS day_of_week,
    TIMESTAMP_DIFF(
        dropoff_datetime,
        pickup_datetime,
        MINUTE
    )                                       AS trip_duration_minutes
FROM
    `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022`
WHERE
    pickup_datetime IS NOT NULL
    AND dropoff_datetime IS NOT NULL
    AND trip_distance > 0
    AND fare_amount > 0
    AND total_amount > 0
    AND passenger_count > 0
    AND passenger_count <= 6
    AND data_file_month = 1
LIMIT 500000
"""

print("⏳ Running query — this takes about 30-60 seconds...")
taxi_df = bq_client.query(taxi_query).to_dataframe()

print(f"✅ Extracted {len(taxi_df):,} taxi records")
print(f"📊 Shape: {taxi_df.shape}")
print(f"📅 Date range: {taxi_df['trip_date'].min()} → {taxi_df['trip_date'].max()}")
print(f"💰 Avg fare: ${taxi_df['fare_amount'].mean():.2f}")
print(f"🚗 Avg distance: {taxi_df['trip_distance'].mean():.2f} miles")
taxi_df.head(3)

🚕 Querying NYC Taxi public dataset...
⏳ Running query — this takes about 30-60 seconds...
✅ Extracted 500,000 taxi records
📊 Shape: (500000, 23)
📅 Date range: 2008-12-31 → 2022-03-15
💰 Avg fare: $27.89
🚗 Avg distance: 8.17 miles


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code,payment_type,fare_amount,extra,mta_tax,...,airport_fee,total_amount,pickup_location_id,dropoff_location_id,data_file_year,data_file_month,trip_date,pickup_hour,day_of_week,trip_duration_minutes
0,1,2022-01-01 00:40:15+00:00,2022-01-01 01:09:48+00:00,1,10.300000000,1.0,1,33.000000000,3.000000000,0.500000000,...,0E-9,56.350000000,138,161,2022,1,2022-01-01,0,7,29
1,2,2022-01-01 00:46:09+00:00,2022-01-01 01:08:06+00:00,6,5.950000000,1.0,1,20.000000000,0.500000000,0.500000000,...,0E-9,28.560000000,79,238,2022,1,2022-01-01,0,7,21
2,2,2022-01-01 18:04:06+00:00,2022-01-01 18:40:52+00:00,1,9.700000000,1.0,1,34.500000000,0.500000000,0.500000000,...,1.250000000,55.320000000,138,48,2022,1,2022-01-01,18,7,36


In [14]:
# Cell 4 - Basic cleaning before landing to raw zone
print("Cleaning data before landing to GCS...")

# Convert datetime columns properly
taxi_df['pickup_datetime'] = pd.to_datetime(taxi_df['pickup_datetime'], utc=True)
taxi_df['dropoff_datetime'] = pd.to_datetime(taxi_df['dropoff_datetime'], utc=True)
taxi_df['trip_date'] = pd.to_datetime(taxi_df['trip_date'])

# Remove any negative fares or distances that slipped through
taxi_df = taxi_df[taxi_df['fare_amount'] > 0]
taxi_df = taxi_df[taxi_df['trip_distance'] > 0]
taxi_df = taxi_df[taxi_df['trip_duration_minutes'] > 0]
taxi_df = taxi_df[taxi_df['trip_duration_minutes'] < 300]

# Add ingestion metadata - critical for pipeline tracking and debugging
taxi_df['ingestion_date'] = INGESTION_DATE
taxi_df['ingestion_timestamp'] = datetime.now().isoformat()
taxi_df['source_system'] = 'bigquery_public_data'
taxi_df['source_table'] = 'tlc_yellow_trips_2022'

print(f"Records after cleaning: {len(taxi_df):,}")
print(f"Columns: {taxi_df.shape[1]}")
print(f"Ingestion date: {INGESTION_DATE}")

Cleaning data before landing to GCS...
Records after cleaning: 491,756
Columns: 27
Ingestion date: 2026-04-28


In [16]:
# Cell 5 - Land cleaned data in GCS raw bucket as partitioned Parquet
print("Uploading to GCS raw bucket...")

def upload_df_to_gcs_parquet(df, bucket_name, gcs_path):
    """
    Upload a pandas DataFrame to GCS as Parquet using an in-memory buffer.
    Avoids writing temporary files to disk.
    """
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(gcs_path)

    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine='pyarrow')
    buffer.seek(0)

    blob.upload_from_file(buffer, content_type='application/octet-stream')
    return f"gs://{bucket_name}/{gcs_path}"

# Partition by ingestion_date
# This ensures late-arriving data never overwrites existing partitions
gcs_path = f"taxi/ingestion_date={INGESTION_DATE}/yellow_trips_jan2022.parquet"

uploaded_path = upload_df_to_gcs_parquet(
    df=taxi_df,
    bucket_name=RAW_BUCKET,
    gcs_path=gcs_path
)

print(f"Upload complete")
print(f"Location: {uploaded_path}")
print(f"Records landed: {len(taxi_df):,}")
print(f"Columns: {taxi_df.shape[1]}")

Uploading to GCS raw bucket...
Upload complete
Location: gs://urban-intelligence-pipeline-sv-raw/taxi/ingestion_date=2026-04-28/yellow_trips_jan2022.parquet
Records landed: 491,756
Columns: 27


In [18]:
# Cell 6 - Verify the landed file by reading it back from GCS
print("Reading back from GCS to verify...")

bucket = storage_client.bucket(RAW_BUCKET)
blob = bucket.blob(gcs_path)

buffer = io.BytesIO()
blob.download_to_file(buffer)
buffer.seek(0)

verify_df = pd.read_parquet(buffer)

print(f"Verification successful")
print(f"Records read back: {len(verify_df):,}")
print(f"Columns: {list(verify_df.columns)}")
print(f"Sample data:")
verify_df[['vendor_id', 'pickup_datetime', 'trip_distance', 
           'fare_amount', 'total_amount', 'ingestion_date']].head(5)

Reading back from GCS to verify...
Verification successful
Records read back: 491,756
Columns: ['vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'rate_code', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'imp_surcharge', 'airport_fee', 'total_amount', 'pickup_location_id', 'dropoff_location_id', 'data_file_year', 'data_file_month', 'trip_date', 'pickup_hour', 'day_of_week', 'trip_duration_minutes', 'ingestion_date', 'ingestion_timestamp', 'source_system', 'source_table']
Sample data:


,vendor_id,pickup_datetime,trip_distance,fare_amount,total_amount,ingestion_date
0,1,2022-01-01 00:40:15+00:00,10.300000000,33.000000000,56.350000000,2026-04-28
1,2,2022-01-01 00:46:09+00:00,5.950000000,20.000000000,28.560000000,2026-04-28
2,2,2022-01-01 18:04:06+00:00,9.700000000,34.500000000,55.320000000,2026-04-28
3,2,2022-01-01 00:55:48+00:00,6.670000000,21.000000000,41.050000000,2026-04-28
4,2,2022-01-01 00:42:45+00:00,6.490000000,19.500000000,28.670000000,2026-04-28
